# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Khuld13/ML-intern-starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%cd /content
!rm -rf ML-intern-starter
!git clone https://github.com/Khuld13/ML-intern-starter.git
%cd ML-intern-starter

import duckdb, numpy as np, pandas as pd
from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
hf_token = userdata.get('HF_TOKEN')
con.sql(f"CREATE OR REPLACE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

REL = 'hf://datasets/FlyRank/internship-warehouse'
for name, month in [('fact_march','2026-03'), ('fact_feb','2026-02'), ('fact_jan','2026-01')]:
    con.sql(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month={month}/data_0.parquet')")
con.sql(f"CREATE OR REPLACE VIEW dim_content AS SELECT * FROM read_parquet('{REL}/dim_content.parquet')")
print("Connected.")

/content
Cloning into 'ML-intern-starter'...
remote: Enumerating objects: 281, done.
remote: Counting objects: 100% (281/281), done.
remote: Compressing objects: 100% (234/234), done.
remote: Total 281 (delta 160), reused 88 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (281/281), 1.97 MiB | 5.55 MiB/s, done.
Resolving deltas: 100% (160/160), done.
/content/ML-intern-starter
Connected.


In [2]:
con.sql(f"""
CREATE OR REPLACE VIEW dim_content AS
SELECT * FROM read_parquet('{REL}/dim_content.parquet');
""")

con.sql(f"""
CREATE OR REPLACE VIEW fact_daily AS
SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03';
""")

print("Connected. Views ready.")

Connected. Views ready.


In [3]:
con.sql("""
    CREATE OR REPLACE VIEW fact_march AS
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet')
""")

con.sql("""
    CREATE OR REPLACE VIEW fact_feb AS
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet')
""")

print("fact_march and fact_feb ready.")

fact_march and fact_feb ready.


In [4]:
con.sql("""
    CREATE OR REPLACE VIEW fact_jan AS
    SELECT * FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-01/data_0.parquet')
""")

jan_features = con.sql("""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) AS impressions_jan,
        SUM(gsc_clicks) AS clicks_jan,
        AVG(gsc_avg_position) AS avg_position_jan,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions_jan,
        STDDEV(gsc_impressions) AS impressions_volatility_jan
    FROM fact_jan
    WHERE gsc_data_available = TRUE
    GROUP BY content_hash_id
""").df()

print("Jan feature rows:", len(jan_features))
print(jan_features.describe())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Jan feature rows: 121544
       impressions_jan     clicks_jan  avg_position_jan  \
count    121544.000000  121544.000000     121544.000000   
mean       1187.674332       3.917429         14.925038   
std        4484.787119      24.524136         16.562830   
min           1.000000       0.000000          0.000000   
25%          12.000000       0.000000          4.963826   
50%         121.000000       0.000000          8.679912   
75%         688.000000       1.000000         18.200629   
max      319883.000000    3382.000000        350.000000   

       days_with_impressions_jan  impressions_volatility_jan  
count              121544.000000               110971.000000  
mean                   19.722430                   21.621852  
std                    12.018744                  139.221313  
min                     1.000000                    0.000000  
25%                     6.000000                    1.310216  
50%                    25.000000                    4.031924  
75

In [5]:
monthly_compare = con.sql("""
    WITH march_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_march
        FROM fact_march
        WHERE gsc_data_available = TRUE
        GROUP BY content_hash_id
    ),
    feb_agg AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS impressions_feb
        FROM fact_feb
        WHERE gsc_data_available = TRUE
        GROUP BY content_hash_id
    )
    SELECT
        m.content_hash_id,
        f.impressions_feb,
        m.impressions_march,
        CASE WHEN m.impressions_march < f.impressions_feb THEN 1 ELSE 0 END AS declined_flag
    FROM march_agg m
    JOIN feb_agg f USING (content_hash_id)
""").df()

# ML-07's volume-floor rule (Signal 2, CONFIRMED)
pop = monthly_compare[monthly_compare['impressions_march'] >= 250].copy()

# Bring in content-level features to model with
dim = con.sql("SELECT * FROM dim_content").df()
df = pop.merge(dim, on='content_hash_id', how='left')

# Exclude: the label itself, the two raw inputs that DEFINE the label,
# and the known-leaky/known-invalid columns from ML-06
leakage_cols = [
    'declined_flag', 'impressions_feb', 'impressions_march',   # label + its direct inputs
    'trend_pct', 'trend_direction', 'is_declining_label',       # ML-06: same fact, 3 forms
    'days_since_update',                                        # ML-06: structurally invalid (July snapshot)
]
candidate_features = [c for c in df.columns if c not in leakage_cols]

print("Rows after volume filter (impressions_march >= 250):", len(df))
print("\nLabel balance (declined_flag):")
print(df['declined_flag'].value_counts(normalize=True))
print("\nCandidate feature count:", len(candidate_features))
print(candidate_features)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows after volume filter (impressions_march >= 250): 68581

Label balance (declined_flag):
declined_flag
0    0.771759
1    0.228241
Name: proportion, dtype: float64

Candidate feature count: 26
['content_hash_id', 'client_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']


In [6]:
import numpy as np
df = df.merge(jan_features, on='content_hash_id', how='left')
df['ctr_jan'] = (df['clicks_jan'] / df['impressions_jan'].replace(0, np.nan))

In [7]:
df['baseline_rule'] = (
    (df['impressions_volatility_jan'] > df['impressions_volatility_jan'].median()) &
    (df['avg_position_jan'] > 10) &
    (df['days_with_impressions_jan'] < df['days_with_impressions_jan'].median())
).astype(int)

In [8]:
import numpy as np

skewed_cols = ['search_volume', 'backlinks', 'cpc']

# Step 1: check how skewed each one actually is, before touching anything
for col in skewed_cols:
    print(col, 'skew before:', df[col].skew())

search_volume skew before: 56.00570972632306
backlinks skew before: 107.44482739749563
cpc skew before: 15.216468946345744


In [9]:
avg_pos_march = con.sql("""
    SELECT content_hash_id, AVG(gsc_avg_position) AS avg_position_march
    FROM fact_march WHERE gsc_data_available = TRUE
    GROUP BY content_hash_id
""").df()
df = df.merge(avg_pos_march, on='content_hash_id', how='left')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [10]:
numeric_features = ['keyword_char_count', 'keyword_token_count', 'url_char_count',
                     'search_volume', 'competition', 'cpc', 'backlinks', 'category_count',
                     'char_count', 'word_count',
                     'impressions_jan', 'clicks_jan', 'avg_position_jan',
                     'days_with_impressions_jan', 'impressions_volatility_jan', 'ctr_jan']
categorical_features = ['content_type', 'competition_level', 'main_intent',
                         'provider_used', 'model_used']

X = df[numeric_features + categorical_features].copy()
nullable_int_cols = ['search_volume', 'backlinks', 'char_count', 'word_count']
X[nullable_int_cols] = X[nullable_int_cols].astype('float64')

skewed_cols = ['search_volume', 'backlinks', 'cpc']
X_log = X.copy()
for col in skewed_cols:
    X_log[col] = np.log1p(X_log[col].clip(lower=0))

y = df['declined_flag']
groups = df['client_hash_id']

In [11]:
preprocessor = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='constant', fill_value='missing')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
])
final_model = Pipeline([('prep', preprocessor), ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))])
final_model.fit(X_log, y)
df['model_risk_score'] = final_model.predict_proba(X_log)[:, 1]

In [12]:
from sklearn.model_selection import cross_val_predict, GroupKFold

gkf = GroupKFold(n_splits=3)
oof_pipeline = Pipeline([('prep', preprocessor), ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))])
oof_scores = cross_val_predict(oof_pipeline, X_log, y, groups=groups, cv=gkf, method='predict_proba')[:, 1]

df['model_risk_score'] = oof_scores
print(df['model_risk_score'].describe())
print(df['model_risk_score'].quantile([0.5, 0.75, 0.9, 0.95, 0.99]))

count    6.858100e+04
mean     4.889051e-01
std      1.723659e-01
min      1.044585e-12
25%      3.871496e-01
50%      5.087062e-01
75%      6.114390e-01
max      1.000000e+00
Name: model_risk_score, dtype: float64
0.50    0.508706
0.75    0.611439
0.90    0.684835
0.95    0.758288
0.99    0.832902
Name: model_risk_score, dtype: float64


In [13]:
df['declining_with_demand'] = ((df['declined_flag'] == 1) & (df['impressions_march'] >= 500)).fillna(False)
df['weak_position_visible'] = ((df['avg_position_march'] > 10) & (df['impressions_march'] >= 250)).fillna(False)
df['thin_visible_page'] = ((df['word_count'] > 0) & (df['word_count'] < 1200) & (df['impressions_march'] >= 250)).fillna(False)
ctr_median_visible = df.loc[df['impressions_jan'] >= 500, 'ctr_jan'].median()
df['low_ctr_visible'] = ((df['ctr_jan'] < ctr_median_visible) & (df['impressions_march'] >= 500)).fillna(False)
vol_p75 = df['impressions_volatility_jan'].quantile(0.75)
df['high_volatility'] = ((df['impressions_volatility_jan'] > vol_p75) & (df['impressions_march'] >= 250)).fillna(False)
df['model_risk_high'] = (df['model_risk_score'] >= 0.65).fillna(False)

def get_reasons(row):
    codes = []
    if row['model_risk_high']: codes.append('model_decline_risk')
    if row['declining_with_demand']: codes.append('declining_with_demand')
    if row['weak_position_visible']: codes.append('page_one_decay_risk')
    if row['thin_visible_page']: codes.append('thin_visible_page')
    if row['low_ctr_visible']: codes.append('low_ctr_visible_page')
    if row['high_volatility']: codes.append('high_volatility')
    return codes

df['reason_codes'] = df.apply(get_reasons, axis=1)

def get_action(row):
    if row['model_risk_high'] and row['declining_with_demand']:
        return 'Full Refresh', 'Critical'
    if row['thin_visible_page'] and (row['declining_with_demand'] or row['weak_position_visible']):
        return 'Expand / Deepen', 'High'
    if row['low_ctr_visible'] and row['weak_position_visible']:
        return 'Metadata Fix', 'High'
    if row['high_volatility']:
        return 'Monitor', 'Medium'
    if len(row['reason_codes']) > 0:
        return 'Review', 'Low'
    return 'No Action', 'None'

df[['action_type', 'priority_tier']] = df.apply(lambda r: pd.Series(get_action(r)), axis=1)

queue = df[df['action_type'] != 'No Action'].sort_values('model_risk_score', ascending=False)
ranked_queue = queue[['content_hash_id', 'action_type', 'priority_tier', 'model_risk_score', 'reason_codes']].head(200)
print(f"Queue size: {len(queue)} of {len(df)} rows flagged for review")
print(ranked_queue.head(10))

Queue size: 52947 of 68581 rows flagged for review
                content_hash_id action_type priority_tier  model_risk_score  \
27276  content_91e8c090a0be842a      Review           Low               1.0   
24265  content_ae287099e8a59058      Review           Low               1.0   
58368  content_44394770fbc4087f      Review           Low               1.0   
27246  content_b7e2e73619eafdf3      Review           Low               1.0   
27262  content_ce6d18b62c6979ea      Review           Low               1.0   
61423  content_e5a09291ae2fcce4      Review           Low               1.0   
58363  content_5e0aafefd9097cad      Review           Low               1.0   
24257  content_c7fd51a0e369d8c4      Review           Low               1.0   
24263  content_48d138a518f82a15      Review           Low               1.0   
68514  content_9ea457a4455e2c7f      Review           Low               1.0   

                                    reason_codes  
27276  [model_decline_risk, 

In [14]:
def get_reasons(row):
    codes = []
    if row['model_risk_high']: codes.append('model_decline_risk')
    if row['declining_with_demand']: codes.append('declining_with_demand')
    if row['weak_position_visible']: codes.append('page_one_decay_risk')
    if row['thin_visible_page']: codes.append('thin_visible_page')
    if row['low_ctr_visible']: codes.append('low_ctr_visible_page')
    if row['high_volatility']: codes.append('high_volatility')
    return codes

df['reason_codes'] = df.apply(get_reasons, axis=1)

def get_action(row):
    if row['model_risk_high'] and row['declining_with_demand']:
        return 'Full Refresh', 'Critical'
    if row['thin_visible_page'] and (row['declining_with_demand'] or row['weak_position_visible']):
        return 'Expand / Deepen', 'High'
    if row['low_ctr_visible'] and row['weak_position_visible']:
        return 'Metadata Fix', 'High'
    if row['high_volatility']:
        return 'Monitor', 'Medium'
    if len(row['reason_codes']) > 0:
        return 'Review', 'Low'
    return 'No Action', 'None'

df[['action_type', 'priority_tier']] = df.apply(lambda r: pd.Series(get_action(r)), axis=1)

In [15]:
df['model_risk_high'] = (df['model_risk_score'] >= 0.65).fillna(False)

In [16]:
reviewable = df[df['reason_codes'].map(len) > 0].sort_values('model_risk_score', ascending=False)
ranked_queue = reviewable.head(100)
print("Action type breakdown in top 100:")
print(ranked_queue['action_type'].value_counts())

Action type breakdown in top 100:
action_type
Full Refresh    46
Monitor         36
Review          18
Name: count, dtype: int64


In [17]:
pd.set_option('display.max_colwidth', None)
print(ranked_queue[['content_hash_id', 'action_type', 'priority_tier', 'model_risk_score', 'reason_codes']].head(20).to_string())

                content_hash_id   action_type priority_tier  model_risk_score                                                                     reason_codes
27276  content_91e8c090a0be842a        Review           Low          1.000000                                        [model_decline_risk, page_one_decay_risk]
24265  content_ae287099e8a59058        Review           Low          1.000000                                                             [model_decline_risk]
58368  content_44394770fbc4087f        Review           Low          1.000000                                                             [model_decline_risk]
27246  content_b7e2e73619eafdf3        Review           Low          1.000000                                        [model_decline_risk, page_one_decay_risk]
27262  content_ce6d18b62c6979ea        Review           Low          1.000000                                                             [model_decline_risk]
61423  content_e5a09291ae2fcce4        Review 

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.